# 01 — Attacker Model

This notebook covers the three modules that define the simulated attacker:

| Module | Contents |
|---|---|
| `attacker/attack_types.py` | Enumerations, severity weights, feature distributions |
| `attacker/transition_model.py` | Intent-shaped Markov chains for attack & stage transitions |
| `attacker/attacker.py` | `Attacker` agent that steps through the kill chain |

All threat categories follow the **UNSW-NB15** dataset taxonomy.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_theme(style='whitegrid')

---
## 1. Core Enumerations (`attack_types.py`)

Three `IntEnum` classes form the vocabulary of the simulation.

In [ ]:
from attacker.attack_types import (
    AttackType,
    KillChainStage,
    AttackerIntent,
    ATTACK_SEVERITY,
    KILL_CHAIN_WEIGHT,
    ATTACK_PRIMARY_STAGE,
    FEATURE_DISTRIBUTIONS,
    FEATURE_NAMES,
)

print('Attack types      :', AttackType.names())
print('Kill chain stages :', KillChainStage.names())
print('Attacker intents  :', AttackerIntent.names())

### 1.1 Attack severity and kill-chain weights

In [ ]:
sev_df = pd.DataFrame({
    'Attack type': AttackType.names(),
    'Severity':    [ATTACK_SEVERITY[int(a)] for a in AttackType],
    'Primary stage': [KillChainStage(ATTACK_PRIMARY_STAGE[int(a)]).name for a in AttackType],
})

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].barh(sev_df['Attack type'], sev_df['Severity'], color='steelblue')
axes[0].set_xlabel('Severity score')
axes[0].set_title('Attack Type Severity')
axes[0].set_xlim(0, 1)

kc_df = pd.DataFrame({
    'Stage':  KillChainStage.names(),
    'Weight': [KILL_CHAIN_WEIGHT[int(s)] for s in KillChainStage],
})
axes[1].barh(kc_df['Stage'], kc_df['Weight'], color='darkorange')
axes[1].set_xlabel('Threat multiplier')
axes[1].set_title('Kill Chain Stage Weights')
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()
sev_df

### 1.2 Network feature distributions

Each attack type has its own parametric distributions for 15 UNSW-NB15 features.  
We sample 300 instances per type and compare key features.

In [ ]:
from attacker.attacker import Attacker

N_SAMPLES = 300
records = []
for attack_type in AttackType:
    attacker = Attacker(seed=0)
    for _ in range(N_SAMPLES):
        feat = attacker._simulate_features(attack_type)
        feat['attack_type'] = attack_type.name
        records.append(feat)

feat_df = pd.DataFrame(records)
feat_df.head()

In [ ]:
key_features = ['dur', 'sload', 'spkts', 'sbytes', 'ct_dst_ltm']

fig, axes = plt.subplots(1, len(key_features), figsize=(18, 6))
fig.suptitle('Feature Distributions by Attack Type', fontsize=13, fontweight='bold')

for i, feat in enumerate(key_features):
    ax = axes[i]
    feat_df.boxplot(column=feat, by='attack_type', ax=ax, vert=True)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45, labelsize=6)
    ax.set_yscale('symlog')
    if i == 0:
        ax.set_ylabel('Value (symlog scale)')

plt.suptitle('Feature Distributions by Attack Type', fontsize=13)
plt.tight_layout()
plt.show()

---
## 2. Transition Model (`transition_model.py`)

The transition model is a **Markov chain** with two separate matrices:
- **Attack matrix** (10×10): next attack type given current  
- **Stage matrix** (7×7): next kill-chain stage given current

Base matrices are multiplied element-wise by intent-specific modifiers, then row-normalized.

In [ ]:
from attacker.transition_model import TransitionModel

n_intents = AttackerIntent.count()
fig, axes = plt.subplots(n_intents, 2, figsize=(18, 5 * n_intents))
fig.suptitle('Transition Matrices by Attacker Intent', fontsize=14, fontweight='bold')

attack_names = AttackType.names()
stage_names  = KillChainStage.names()

for row, intent in enumerate(AttackerIntent):
    tm = TransitionModel(intent=intent)

    # Attack type transitions
    ax = axes[row, 0]
    sns.heatmap(
        tm.get_attack_matrix(), annot=True, fmt='.2f', cmap='Blues',
        xticklabels=attack_names, yticklabels=attack_names,
        ax=ax, vmin=0, vmax=1, cbar=False, annot_kws={'size': 6},
    )
    ax.set_title(f'[{intent.name}] Attack Type Transitions', fontsize=10)
    ax.set_xlabel('Next Attack')
    ax.set_ylabel('Current Attack')
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

    # Kill chain stage transitions
    ax = axes[row, 1]
    sns.heatmap(
        tm.get_stage_matrix(), annot=True, fmt='.2f', cmap='Greens',
        xticklabels=stage_names, yticklabels=stage_names,
        ax=ax, vmin=0, vmax=1, cbar=False, annot_kws={'size': 8},
    )
    ax.set_title(f'[{intent.name}] Kill Chain Stage Transitions', fontsize=10)
    ax.set_xlabel('Next Stage')
    ax.set_ylabel('Current Stage')
    ax.tick_params(axis='x', rotation=30, labelsize=7)
    ax.tick_params(axis='y', rotation=0,  labelsize=7)

plt.tight_layout()
plt.show()

---
## 3. Attacker Agent (`attacker.py`)

`Attacker` wraps a `TransitionModel` and generates a step-by-step sequence of
attack events, each with simulated network features.

In [ ]:
# Run 60 steps with the STEALTHY intent and record the trajectory
attacker = Attacker(intent=AttackerIntent.STEALTHY, seed=7)
attacker.reset()

steps, attack_ids, stage_ids = [], [], []
for t in range(60):
    info = attacker.step()
    steps.append(t)
    attack_ids.append(int(info['attack_type']))
    stage_ids.append(int(info['kill_chain_stage']))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle('Attacker Trajectory — STEALTHY intent (60 steps)', fontsize=12)

ax1.step(steps, attack_ids, where='post', color='steelblue')
ax1.set_yticks(range(AttackType.count()))
ax1.set_yticklabels(AttackType.names(), fontsize=8)
ax1.set_ylabel('Attack Type')
ax1.grid(True, alpha=0.3)

ax2.step(steps, stage_ids, where='post', color='darkorange')
ax2.set_yticks(range(KillChainStage.count()))
ax2.set_yticklabels(KillChainStage.names(), fontsize=8)
ax2.set_ylabel('Kill Chain Stage')
ax2.set_xlabel('Step')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare attack-type distributions across all 4 intents
N = 500
counts = {}
for intent in AttackerIntent:
    attacker = Attacker(intent=intent, seed=42)
    attacker.reset()
    freq = {name: 0 for name in AttackType.names()}
    for _ in range(N):
        info = attacker.step()
        freq[info['attack_type'].name] += 1
    counts[intent.name] = freq

dist_df = pd.DataFrame(counts).T
dist_df.div(N).plot(kind='bar', figsize=(14, 5), width=0.7)
plt.title('Attack Type Frequency by Attacker Intent (500 steps)')
plt.ylabel('Proportion')
plt.xticks(rotation=0)
plt.legend(loc='upper right', fontsize=7)
plt.tight_layout()
plt.show()